In [1]:
import os
import pandas as pd
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

Loading the dataset

In [2]:
# =====================================================================
# 1. PARAMETERS & PATH CONFIGURATIONS
# =====================================================================
# Cross-folder paths to step out of 'ml/' and find 'ESC-50-master'
AUDIO_DIR = "../ESC-50-master/audio/"  
CSV_PATH = "../ESC-50-master/meta/esc50.csv"  

SR = 22050                               # Standardized audio sample rate
DURATION = 5                             # ESC-50 clips are exactly 5 seconds long
N_MELS = 64                              # Number of Mel frequency bands
MAX_PAD_LEN = 216                        # Fixed horizontal width dimension for the CNN matrix

# Target classes matching your notebook exactly
IMPORTANT_CLASSES = [
    'siren',
    'crying_baby',
    'door_wood_knock',
    'glass_breaking',
    'fireworks',
    'clock_alarm',
    'car_horn',
    'train'
]

# Structural label map converting text categories to numerical targets
LABEL_MAP = {category: idx for idx, category in enumerate(IMPORTANT_CLASSES)}

In [9]:
# =====================================================================
# 2. DIGITAL SIGNAL PROCESSING & AUGMENTATION ENGINE
# =====================================================================
def process_audio_to_mel(audio, sr, n_mels=N_MELS, max_pad_len=MAX_PAD_LEN):
    """Converts a raw audio array into a perfectly padded 2D Mel Spectrogram."""
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Padding/Trimming loop
    if mel_spec_db.shape[1] < max_pad_len:
        pad_width = max_pad_len - mel_spec_db.shape[1]
        mel_spec_db = np.pad(mel_spec_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        mel_spec_db = mel_spec_db[:, :max_pad_len]
        
    return mel_spec_db

def add_noise(data, noise_factor=0.005):
    """Injects random white noise into the audio waveform."""
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

def pitch_shift(data, sr, n_steps=2):
    """Shifts the audio pitch up or down slightly without changing speed."""
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=n_steps)

Filtering the important classes

In [19]:
pip install resampy

   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 20.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
# =====================================================================
# 3. PIPELINE PREPARATION & DATASET MULTIPLIER
# =====================================================================
print("🔄 Parsing metadata index file...")
df = pd.read_csv(CSV_PATH)
df_filtered = df[df['category'].isin(IMPORTANT_CLASSES)].reset_index(drop=True)

X, y = [], []

print(f"📦 Processing & Augmenting Audio Files (Tripling the dataset...)")
success_count = 0

for _, row in df_filtered.iterrows():
    full_audio_path = os.path.join(AUDIO_DIR, row['filename'])
    
    try:
        # Load the base audio exactly ONCE to save processing time
        audio, sr = librosa.load(full_audio_path, sr=SR, res_type='kaiser_fast')
        label = LABEL_MAP[row['category']]
        
        # --- 1. The Original Audio ---
        mel_orig = process_audio_to_mel(audio, sr)
        if mel_orig is not None:
            X.append(mel_orig)
            y.append(label)
            
        # --- 2. The Noise-Injected Audio ---
        audio_noise = add_noise(audio)
        mel_noise = process_audio_to_mel(audio_noise, sr)
        if mel_noise is not None:
            X.append(mel_noise)
            y.append(label)
            
        # --- 3. The Pitch-Shifted Audio ---
        audio_pitch = pitch_shift(audio, sr, n_steps=2) # Shift up by 2 half-steps
        mel_pitch = process_audio_to_mel(audio_pitch, sr)
        if mel_pitch is not None:
            X.append(mel_pitch)
            y.append(label)

        success_count += 1
        
    except Exception as e:
        print(f"⚠️ Error processing file {full_audio_path}: {e}")

print(f"✅ Preprocessing complete! Successfully augmented {success_count} original files.")

X = np.array(X)
y = np.array(y)

# Format shape for CNN: (Samples, Height, Width, Channels)
X_cnn = X.reshape(-1, X.shape[1], X.shape[2], 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_cnn, y, test_size=0.2, random_state=42, stratify=y
)
print(f"🚀 MASSIVE DATASET READY! Train shape: {X_train.shape}, Validation shape: {X_val.shape}")

🔄 Parsing metadata index file...
📦 Processing & Augmenting Audio Files (Tripling the dataset...)
✅ Preprocessing complete! Successfully augmented 320 original files.
🚀 MASSIVE DATASET READY! Train shape: (768, 64, 216, 1), Validation shape: (192, 64, 216, 1)


EDA

In [11]:
# =====================================================================
# 4. CONVOLUTIONAL NEURAL NETWORK ARCHITECTURE
# =====================================================================
model = models.Sequential([
    layers.Input(shape=(N_MELS, MAX_PAD_LEN, 1)),
    
    # Layer Block 1
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    
    # Layer Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    
    # Dense Classifier Head
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),  # Prevents overfitting on smaller sample pools
    layers.Dense(len(IMPORTANT_CLASSES), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [7]:
# A function that helps us to load the audio in wav plot
import IPython.display as ipd

ipd.Audio(filename)

In [12]:
# =====================================================================
# 5. MODEL TRAINING LOOP (WITH EARLY STOPPING)
# =====================================================================
print("\n🚀 Starting model training loop...")

# This acts as a referee: it watches the validation loss and stops the fight if the model stops improving
early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=8,                # Wait 8 epochs to see if it recovers
    restore_best_weights=True  # Automatically keep the best version!
)

history = model.fit(
    X_train, y_train,
    epochs=100,                # Give it plenty of room to run
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stopper]  # Activate the referee
)


🚀 Starting model training loop...
Epoch 1/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.4537 - loss: 6.9200 - val_accuracy: 0.1771 - val_loss: 29.3682
Epoch 2/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.7035 - loss: 2.1011 - val_accuracy: 0.6823 - val_loss: 5.0558
Epoch 3/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.7848 - loss: 0.8499 - val_accuracy: 0.8333 - val_loss: 0.5409
Epoch 4/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.8487 - loss: 1.2607 - val_accuracy: 0.8646 - val_loss: 0.3687
Epoch 5/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.8781 - loss: 0.5277 - val_accuracy: 0.8698 - val_loss: 0.5363
Epoch 6/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.8638 - loss: 0.5131 - val_accuracy: 0.8646 - val_loss: 0.6052
Epoch 7/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.8639 - loss: 0.4937 - val_accuracy: 0.9427 - val_loss: 0.1711
Epoch 8/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.91

In [13]:
# =====================================================================
# 6. TENSORFLOW LITE CONVERSION & QUANTIZATION
# =====================================================================
print("\n⚙️ Quantizing weights and converting model to an optimized .tflite format...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Standard model compression optimization
tflite_model = converter.convert()

# Export model binary asset
tflite_filename = 'emergency_audio_classifier.tflite'
with open(tflite_filename, 'wb') as f:
    f.write(tflite_model)

# Export complementary Flutter configuration text tracking labels
labels_filename = 'labels.txt'
with open(labels_filename, 'w') as f:
    for category in IMPORTANT_CLASSES:
        f.write(f"{category}\n")

print(f"\n🎉 SUCCESS! Local production assets compiled cleanly inside your ml folder:")
print(f"   1️⃣ Model Binary Asset -> '{tflite_filename}'")
print(f"   2️⃣ Flutter Map Index Label Asset -> '{labels_filename}'")


⚙️ Quantizing weights and converting model to an optimized .tflite format...
INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmp4lbbt1dg\assets


INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmp4lbbt1dg\assets


Saved artifact at 'C:\Users\ACER\AppData\Local\Temp\tmp4lbbt1dg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 216, 1), dtype=tf.float32, name='keras_tensor_11')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  1488997318096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1488997318288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357178256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357177872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357177296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357177680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357179792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357180368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357180560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489357179408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1489